In [0]:
# Cell G — Gold: business aggregates
from pyspark.sql import functions as F

df_silver = spark.table("silver_yellow_taxi")

# 1. Daily trip volume & revenue
gold_daily_summary = (
    df_silver
    .withColumn("trip_date", F.to_date("tpep_pickup_datetime"))
    .groupBy("trip_date")
    .agg(
        F.count("*").alias("trip_count"),
        F.sum("fare_amount").alias("total_fare"),
        F.sum("tip_amount").alias("total_tip"),
        F.sum("total_amount").alias("total_revenue"),
        F.avg("trip_distance").alias("avg_trip_distance"),
        F.avg("trip_duration_min").alias("avg_trip_duration_min")
    )
    .orderBy("trip_date")
)
gold_daily_summary.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.gold_daily_summary")

# 2. Fare/tip patterns by hour of day and day of week
gold_hourly_patterns = (
    df_silver
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_dow", F.date_format("tpep_pickup_datetime", "EEEE"))
    .withColumn("pickup_dow_num", F.dayofweek("tpep_pickup_datetime"))  # 1=Sun...7=Sat, sort key only
    .groupBy("pickup_dow", "pickup_dow_num", "pickup_hour")
    .agg(
        F.count("*").alias("trip_count"),
        F.avg("fare_amount").alias("avg_fare"),
        F.avg("tip_amount").alias("avg_tip"),
        F.avg(F.when(F.col("fare_amount") > 0, F.col("tip_amount") / F.col("fare_amount"))).alias("avg_tip_pct")
    )
    .orderBy("pickup_dow_num", "pickup_hour")
)
gold_hourly_patterns.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.gold_hourly_patterns")

# 3. Busiest pickup zones
gold_zone_activity = (
    df_silver
    .groupBy("PULocationID", "PUBorough", "PUZone")
    .agg(
        F.count("*").alias("pickup_count"),
        F.sum("total_amount").alias("total_revenue"),
        F.avg("tip_amount").alias("avg_tip")
    )
    .orderBy(F.desc("pickup_count"))
)
gold_zone_activity.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.gold_zone_activity")

# 4. Payment type mix (raw codes mapped to labels per TLC data dictionary)
payment_labels = {1: "Credit card", 2: "Cash", 3: "No charge", 4: "Dispute", 5: "Unknown", 6: "Voided trip"}
mapping_expr = F.create_map([F.lit(x) for pair in payment_labels.items() for x in pair])

gold_payment_mix = (
    df_silver
    .withColumn("payment_label", F.coalesce(mapping_expr[F.col("payment_type")], F.lit("Other/Unrecognized")))
    .groupBy("payment_label")
    .agg(
        F.count("*").alias("trip_count"),
        F.sum("total_amount").alias("total_revenue"),
        F.avg("tip_amount").alias("avg_tip")
    )
    .orderBy(F.desc("trip_count"))
)
gold_payment_mix.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.gold_payment_mix")

# 5. Borough-to-borough flow
gold_borough_flow = (
    df_silver
    .groupBy("PUBorough", "DOBorough")
    .agg(
        F.count("*").alias("trip_count"),
        F.avg("trip_distance").alias("avg_distance"),
        F.avg("total_amount").alias("avg_fare")
    )
    .orderBy(F.desc("trip_count"))
)
gold_borough_flow.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.gold_borough_flow")

display(gold_daily_summary)

trip_date,trip_count,total_fare,total_tip,total_revenue,avg_trip_distance,avg_trip_duration_min
2008-12-31,1,17.7,4.49,26.94,2.28,1141.4
2009-01-01,1,79.3,12.0,110.97,13.01,95.0
2026-04-30,11,266.79999999999995,26.9,373.65999999999997,4.83,18.986363636363635
2026-05-01,131182,2785591.909999896,404471.9099999993,3988956.209999801,3.5618245643457125,19.675196292174256
2026-05-02,132055,2639793.3099998333,358945.7699999982,3721159.139999722,4.556117375336046,17.084924968131862
2026-05-03,118189,2613314.8199998224,341480.6999999984,3645385.189999703,7.557790826557451,16.723600053022388
2026-05-04,102028,2230017.109999958,348163.09000000195,3208527.7599998964,5.281102834515999,19.89326263378666
2026-05-05,123406,2589897.48999992,392812.4400000002,3732231.0899998248,4.531088763917502,19.165189969153282
2026-05-06,132583,2876711.899999841,422875.1300000014,4098088.6799997385,6.735360566588533,19.651483347538353
2026-05-07,131736,2853578.4999998696,436923.5500000029,4089339.7599997744,3.8296798900831788,20.14852976154322


In [0]:
# Fallback — export Gold tables to CSV in your volume
table_path = "/Volumes/workspace/default/raw_data"
gold_tables = ["gold_daily_summary", "gold_hourly_patterns", "gold_zone_activity", "gold_payment_mix", "gold_borough_flow"]

for table in gold_tables:
    spark.table(table).toPandas().to_csv(f"{table_path}/{table}.csv", index=False)

print("Exported. In the Databricks sidebar: Catalog > workspace > default > raw_data — download each CSV, then Get Data > Text/CSV in Power BI Desktop.")

Exported. In the Databricks sidebar: Catalog > workspace > default > raw_data — download each CSV, then Get Data > Text/CSV in Power BI Desktop.
